# Notebook 13D — Policy A sensitivity analysis

**Question:** Does the retraining result change if we use the more conservative Policy A training population instead of Policy B?

Notebook 13 trained an updated XGBoost using **Policy B** (mature individual loans), and Notebooks 13B–13C evaluated and diagnosed that model. Here we fit **Policy A** (whole mature half-year cohorts), then compare three models on the **same 2023 loans**:

- **Frozen:** original 2015–2017 XGBoost.
- **Policy A:** 2015–2017 development loans plus 2018-H1 and 2018-H2.
- **Policy B:** the updated XGBoost already saved in Notebook 13.

We reuse the fitted 2015–2017 preprocessing, the original 29 predictors, and the original XGBoost settings. Neither existing model is changed. This is a **retrospective exploratory sensitivity analysis**, not an independent prospective test.

## 1. Verify the original model, Policy B model and audit files

Confirm the saved model hashes and preprocessing contract before using them. We also confirm that the original and Policy B XGBoost settings match. The January 2023 decision and training-policy rules stay fixed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
import sys, json, hashlib, zipfile
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.freddie_features import (
    REQUIRED_SOURCE_COLUMNS, PREDICTOR_COLUMNS, NUMERIC_FEATURES,
    CATEGORICAL_FEATURES, build_engineered_features, apply_preprocessing_contract,
)

DATA = ROOT / 'data' / 'processed' / 'Freddie_Mac'
BASE = DATA / 'model_baselines_2015_2017'
RUN_ID_B = '20261111T161902005765Z'  # Stage 2 Policy B run, reviewed in Notebook 13B
B_DIR = DATA / 'notebook_13_updated_xgboost' / RUN_ID_B
REFERENCE_DIR = DATA / 'notebook_13_retraining_evaluation' / RUN_ID_B
AUDIT_ROOT = DATA / 'notebook_13_retraining_policy_comparison'
MODEL_DATA = DATA / 'freddie_modeling_dataset_2015_2023_24m.parquet'
BASE_MANIFEST = BASE / 'model_manifest.json'
CONTRACT_PATH = BASE / 'freddie_feature_contract.json'
B_MANIFEST = B_DIR / 'updated_model_manifest.json'
SAVED_PREDICTIONS = BASE / 'oot_evaluation' / 'oot_predictions_2018_2023.parquet'
REFERENCE_METRICS = REFERENCE_DIR / 'frozen_updated_metrics.csv'
REFERENCE_MANIFEST = REFERENCE_DIR / 'evaluation_manifest.json'
KEYS = ['SOURCE YEAR', 'LOAN IDENTIFIER']
TARGET = 'TARGET_COMPOSITE_CREDIT_EVENT'


def require(ok, message):
    if not ok:
        raise ValueError(message)


def sha256(path):
    with Path(path).open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def same_setting(a, b):
    if a is None or b is None:
        return a is b
    if isinstance(a, (float, np.floating)) and isinstance(b, (float, np.floating)):
        if np.isnan(a) and np.isnan(b):
            return True
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return bool(np.array_equal(a, b, equal_nan=True))
    return bool(a == b)


needed = [MODEL_DATA, BASE_MANIFEST, CONTRACT_PATH, B_MANIFEST,
          SAVED_PREDICTIONS, REFERENCE_METRICS, REFERENCE_MANIFEST]
require(all(path.is_file() for path in needed),
        'Required Notebook 10, 11, 13 or 13B files are missing.')
base_info = json.loads(BASE_MANIFEST.read_text(encoding='utf-8'))
contract = json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
b_info = json.loads(B_MANIFEST.read_text(encoding='utf-8'))
reference_info = json.loads(REFERENCE_MANIFEST.read_text(encoding='utf-8'))
frozen_path = BASE / base_info['models']['xgboost']['artifact']
b_path = B_DIR / b_info['model_filename']
require(frozen_path.is_file() and b_path.is_file(), 'A saved model file is missing.')
require(sha256(MODEL_DATA) == base_info['input_file_sha256'], 'Modeling data changed.')
require(sha256(CONTRACT_PATH) == base_info['feature_contract_sha256']
        == b_info['original_feature_contract_sha256'], 'Frozen contract changed.')
require(sha256(frozen_path) == base_info['models']['xgboost']['sha256']
        == b_info['frozen_model_sha256'], 'Frozen model changed.')
require(sha256(b_path) == b_info['updated_model_sha256'], 'Policy B model changed.')
require(base_info['predictors'] == PREDICTOR_COLUMNS
        and base_info['numeric_predictors'] == NUMERIC_FEATURES
        and base_info['categorical_predictors'] == CATEGORICAL_FEATURES
        and len(PREDICTOR_COLUMNS) == 29, 'The 29 predictor contract differs.')
require(contract['training_years'] == [2015, 2016, 2017],
        'The original preprocessing years differ.')
require(b_info['primary_training_policy'] == 'B_full_window_individual'
        and b_info['decision_month'] == '2023-01'
        and b_info['sample_rows'] == 285070
        and b_info['sample_events'] == 5179,
        'Policy B experiment differs from the reviewed version.')
require(reference_info['evaluation_loans'] == 49501
        and reference_info['evaluation_events'] == 876
        and reference_info['frozen_model_sha256'] == sha256(frozen_path)
        and reference_info['updated_model_sha256'] == sha256(b_path),
        'Notebook 13B evaluated different loans or models.')

frozen_model = joblib.load(frozen_path)
b_model = joblib.load(b_path)
require(isinstance(frozen_model, Pipeline)
        and isinstance(b_model, Pipeline)
        and isinstance(frozen_model.named_steps['model'], XGBClassifier)
        and isinstance(b_model.named_steps['model'], XGBClassifier),
        'Unexpected saved model type.')
base_params = frozen_model.named_steps['model'].get_params()
b_params = b_model.named_steps['model'].get_params()
settings = sorted(set(base_params) | set(b_params))
differences = [key for key in settings if key not in base_params or key not in b_params
               or not same_setting(base_params[key], b_params[key])]
require(not differences, f'Frozen/Policy B XGBoost settings differ: {differences}')
for key, value in base_info['models']['xgboost']['selected_params'].items():
    require(same_setting(base_params.get(key), value), f'Original setting changed: {key}')
np.testing.assert_array_equal(
    frozen_model.named_steps['prepare'].get_feature_names_out(),
    b_model.named_steps['prepare'].get_feature_names_out(),
)
print('Verified frozen model, Policy B model and saved 13B evaluation references.')
print('Both models use the original XGBoost settings and feature definitions.')


## 2. Reconstruct and verify the Policy A training population

Policy A retains all **2015–2017** development loans and adds **2018-H1 and 2018-H2** only. The January 2023 policy audit identified these as the fully mature whole cohorts. We verify the selected loan-key fingerprint against the saved Stage 1B policy audit, including its maturity checks.

The data contain historical outcomes, but selection here uses only the **predefined cohort rule**. We do not select individual loans because of their target or their 2023 performance.

In [ ]:
columns = list(dict.fromkeys([*REQUIRED_SOURCE_COLUMNS, 'HORIZON_MONTHS']))
raw = pd.read_parquet(MODEL_DATA, columns=columns)
raw['SOURCE YEAR'] = pd.to_numeric(raw['SOURCE YEAR'], errors='raise').astype(int)
raw['LOAN IDENTIFIER'] = raw['LOAN IDENTIFIER'].astype('string')
require(len(raw) == 437668 and not raw.duplicated(KEYS).any(),
        'Modeling dataset or key totals differ.')
require(raw['HORIZON_MONTHS'].eq(24).all(), 'Target horizon changed.')
require(raw[TARGET].isin([0, 1]).all(), 'Target is not binary.')
require(raw['SOURCE YEAR'].between(2015, 2023).all(), 'Unexpected source year.')

q = pd.to_numeric(raw['ORIGINATION QUARTER'].astype('string').str.strip()
                  .str.removeprefix('Q'), errors='raise').astype(int)
require(q.isin([1, 2, 3, 4]).all(), 'Unexpected quarter value.')
require(raw['SOURCE YEAR'].eq(pd.to_numeric(raw['ORIGINATION YEAR'], errors='raise')).all(),
        'Source year and origination year differ.')
raw['period'] = raw['SOURCE YEAR'].astype(str) + '-' + np.where(q.le(2), 'H1', 'H2')

train_a = raw.loc[raw['SOURCE YEAR'].isin([2015, 2016, 2017, 2018])].copy()
raw_2023 = raw.loc[raw['SOURCE YEAR'].eq(2023)].copy()
del raw
require(not train_a['LOAN IDENTIFIER'].isin(raw_2023['LOAN IDENTIFIER']).any(),
        'Train and evaluation loan identifiers overlap.')

cohort_a = (train_a.groupby('period', sort=True)
    .agg(loans=(TARGET, 'size'), events=(TARGET, 'sum')).reset_index())
expected_a = {
    '2015-H1': (23602, 87), '2015-H2': (23642, 161),
    '2016-H1': (24039, 157), '2016-H2': (24037, 167),
    '2017-H1': (24091, 222), '2017-H2': (24187, 169),
    '2018-H1': (23699, 327), '2018-H2': (23718, 895),
}
require(set(cohort_a['period']) == set(expected_a), 'Policy A cohort selection differs.')
for row in cohort_a.itertuples(index=False):
    require((int(row.loans), int(row.events)) == expected_a[row.period],
            f'Policy A cohort totals differ in {row.period}.')
require(len(train_a) == 191015 and int(train_a[TARGET].sum()) == 2185,
        'Policy A training totals differ from Stage 1.')

# The exact loan-key fingerprint was created in Notebook 13 Stage 1B.
normalized_keys = train_a[KEYS].sort_values(KEYS).astype(str)
key_text = '\n'.join(normalized_keys.agg('|'.join, axis=1))
a_key_hash = hashlib.sha256(key_text.encode('utf-8')).hexdigest()

reviewed_audits = []
for spec_path in sorted(AUDIT_ROOT.glob('*/maturity_policy_specification.json')):
    policy_spec = json.loads(spec_path.read_text(encoding='utf-8'))
    fingerprints = policy_spec.get('loan_key_fingerprints_sha256', {})
    if (policy_spec.get('decision_month') == '2023-01'
        and policy_spec.get('audit_status') == 'structural_checks_passed_population_review_required'
        and fingerprints.get('A_whole_cohort') == a_key_hash
        and fingerprints.get('B_full_window_individual') == b_info['selected_loan_keys_sha256']):
        checks_path = spec_path.parent / 'policy_feasibility_checks.csv'
        require(checks_path.is_file(), f'Missing policy checks: {checks_path}')
        audit_checks = pd.read_csv(checks_path)
        require(set(audit_checks['policy']) ==
                {'A_whole_cohort', 'B_full_window_individual'},
                'Audit policy checks do not cover both policies.')
        require(audit_checks.groupby('policy').size().min() > 0, 'Missing audit checks.')
        require(audit_checks['passed'].astype(str).str.lower().eq('true').all(),
                'A policy-audit feasibility check failed.')
        reviewed_audits.append(spec_path)
require(reviewed_audits, 'No matching passed Stage 1B policy audit found; do not fit.')
selected_audit = reviewed_audits[-1]

cohort_a['non_events'] = cohort_a['loans'] - cohort_a['events']
display(cohort_a)
print('Policy A training loans:', len(train_a), '| events:', int(train_a[TARGET].sum()))
print('Policy A selection fingerprint matches audited whole-cohort policy.')
print('Verified audit:', selected_audit)


## 3. Fit a Policy A XGBoost without changing the original preprocessing

Use the same fitted encoder and feature contract as the frozen model. Clone **only** the original XGBoost estimator, keep all its settings, and fit it on the Policy A training loans. This is the sensitivity experiment; there is no new hyperparameter search, weighting or recalibration.

In [ ]:
engineered_a = build_engineered_features(train_a)
prepared_a = apply_preprocessing_contract(engineered_a, contract)
X_a = prepared_a[PREDICTOR_COLUMNS].copy()
y_a = prepared_a[TARGET].astype(int).to_numpy()
require(len(X_a) == 191015 and int(y_a.sum()) == 2185
        and set(np.unique(y_a)) == {0, 1}, 'Policy A labels differ.')
require(not X_a.isna().any().any()
        and np.isfinite(prepared_a[NUMERIC_FEATURES].to_numpy(dtype=float)).all(),
        'Prepared Policy A predictors contain missing or infinite values.')

# Preserve the *fitted* 2015-2017 preprocessing rather than learning new levels.
prepare_a = deepcopy(frozen_model.named_steps['prepare'])
model_a = clone(frozen_model.named_steps['model'])
params_a = model_a.get_params()
require(set(params_a) == set(base_params)
        and all(same_setting(params_a[k], base_params[k]) for k in params_a),
        'Cloned XGBoost settings differ from the baseline.')
X_a_encoded = prepare_a.transform(X_a)
require(X_a_encoded.shape[0] == len(y_a), 'Encoded sample size differs.')
model_a.fit(X_a_encoded, y_a)
policy_a_model = Pipeline([('prepare', prepare_a), ('model', model_a)])

probe = X_a.head(50)
np.testing.assert_allclose(
    model_a.predict_proba(prepare_a.transform(probe))[:, 1],
    policy_a_model.predict_proba(probe)[:, 1], rtol=1e-12, atol=1e-12,
)
np.testing.assert_array_equal(
    policy_a_model.named_steps['prepare'].get_feature_names_out(),
    frozen_model.named_steps['prepare'].get_feature_names_out(),
)
print('Policy A XGBoost fit completed (same original settings and preprocessing).')
print('Policy A training loans:', len(y_a), '| credit events:', int(y_a.sum()))
print('No frozen or Policy B model has been changed or replaced.')


## 4. Score exactly the same 2023 loans

Both updated models and the frozen model now predict the same 2023-H1 and 2023-H2 loans. Reproduce the frozen Notebook 11 scores and the original Notebook 13B metrics before calculating new comparisons.

In [ ]:
engineered_2023 = build_engineered_features(raw_2023)
prepared_2023 = apply_preprocessing_contract(engineered_2023, contract)
require(not prepared_2023[PREDICTOR_COLUMNS].isna().any().any()
        and np.isfinite(prepared_2023[NUMERIC_FEATURES].to_numpy(dtype=float)).all(),
        '2023 preprocessing produced missing or invalid predictors.')
X_2023 = prepared_2023[PREDICTOR_COLUMNS].copy()
evaluation = prepared_2023[KEYS + ['ORIGINATION QUARTER', TARGET]].copy()
evaluation['period'] = np.where(
    pd.to_numeric(evaluation['ORIGINATION QUARTER'].astype('string').str.strip()
                  .str.removeprefix('Q'), errors='raise').astype(int).le(2),
    '2023-H1', '2023-H2',
)
evaluation = evaluation.drop(columns='ORIGINATION QUARTER').rename(columns={TARGET:'y_true'})
evaluation['p_frozen'] = frozen_model.predict_proba(X_2023)[:, 1]
evaluation['p_policy_a'] = policy_a_model.predict_proba(X_2023)[:, 1]
evaluation['p_policy_b'] = b_model.predict_proba(X_2023)[:, 1]
require(not evaluation.duplicated(KEYS).any(), 'Duplicate evaluation loan keys.')
for name in ['p_frozen', 'p_policy_a', 'p_policy_b']:
    require(np.isfinite(evaluation[name]).all() and evaluation[name].between(0, 1).all(),
            f'Invalid probabilities: {name}')

saved = pd.read_parquet(SAVED_PREDICTIONS)
saved['SOURCE YEAR'] = pd.to_numeric(saved['SOURCE YEAR'], errors='raise').astype(int)
saved['LOAN IDENTIFIER'] = saved['LOAN IDENTIFIER'].astype('string')
saved = saved.loc[saved['SOURCE YEAR'].eq(2023),
                  KEYS + ['y_true', 'half_year', 'p_xgboost']]
require(not saved.duplicated(KEYS).any(), 'Saved Notebook 11 keys are duplicated.')
merged = evaluation.merge(saved, on=KEYS, how='outer', validate='one_to_one',
                          suffixes=('', '_saved'), indicator=True)
require(merged['_merge'].eq('both').all()
        and merged['y_true'].eq(merged['y_true_saved']).all()
        and merged['period'].eq(merged['half_year']).all(),
        'Notebook 11 evaluation keys, periods or labels do not reconcile.')
np.testing.assert_allclose(merged['p_frozen'], merged['p_xgboost'],
                           rtol=1e-6, atol=1e-8)
evaluation = merged[KEYS + ['period', 'y_true',
                            'p_frozen', 'p_policy_a', 'p_policy_b']]
evaluation = evaluation.sort_values(KEYS).reset_index(drop=True)
cohort_eval = evaluation.groupby('period', sort=True).agg(
    loans=('y_true', 'size'), events=('y_true', 'sum')).reset_index()
expected_eval = {'2023-H1': (24757, 460), '2023-H2': (24744, 416)}
require(set(cohort_eval['period']) == set(expected_eval), '2023 cohorts differ.')
for row in cohort_eval.itertuples(index=False):
    require((int(row.loans), int(row.events)) == expected_eval[row.period],
            f'{row.period}: evaluation totals differ.')
require(len(evaluation) == 49501 and int(evaluation['y_true'].sum()) == 876,
        'The 2023 evaluation population differs.')
display(cohort_eval)
print('Same 2023 loans for all three models. Frozen scores match Notebook 11.')


## 5. Compare ROC-AUC, Average Precision, Brier Score and calibration

Higher ROC-AUC and Average Precision are better; lower Brier Score and absolute risk gap are better. We show each half-year and all of 2023. A **positive improvement** favors the **first** model in a pair. The earlier Policy B versus frozen results must reproduce Notebook 13B.

In [ ]:
def score_metrics(y, p):
    return {
        'roc_auc': float(roc_auc_score(y, p)),
        'average_precision': float(average_precision_score(y, p)),
        'brier': float(brier_score_loss(y, p)),
        'risk_gap': float(y.mean() - p.mean()),
        'absolute_risk_gap': float(abs(y.mean() - p.mean())),
        'mean_predicted_risk': float(p.mean()),
    }

METRICS = ['roc_auc', 'average_precision', 'brier', 'absolute_risk_gap']
HIGHER_BETTER = {'roc_auc', 'average_precision'}
MODEL_NAMES = ['frozen', 'policy_a', 'policy_b']
PAIRS = [
    ('policy_a', 'frozen'),
    ('policy_b', 'frozen'),
    ('policy_a', 'policy_b'),
]
GROUPS = {
    '2023-H1': evaluation.loc[evaluation['period'].eq('2023-H1')],
    '2023-H2': evaluation.loc[evaluation['period'].eq('2023-H2')],
    '2023-all': evaluation,
}

rows, contrasts = [], []
for period, part in GROUPS.items():
    y = part['y_true'].to_numpy(dtype=int)
    require(np.unique(y).size == 2, 'ROC-AUC requires both classes.')
    scores = {}
    for name in MODEL_NAMES:
        scores[name] = score_metrics(y, part[f'p_{name}'].to_numpy(dtype=float))
        rows.append({'period': period, 'model': name,
                     'loans': len(y), 'events': int(y.sum()),
                     'prevalence': float(y.mean()), **scores[name]})
    for first, second in PAIRS:
        for metric in METRICS:
            a, b = scores[first][metric], scores[second][metric]
            improvement = a-b if metric in HIGHER_BETTER else b-a
            contrasts.append({'period': period, 'comparison': f'{first}_minus_{second}',
                              'metric': metric, 'first': a, 'second': b,
                              'improvement': improvement})
model_metrics = pd.DataFrame(rows)
point_comparisons = pd.DataFrame(contrasts)

# Independent reproduction of the saved 13B frozen / Policy B metrics.
reference = pd.read_csv(REFERENCE_METRICS)
reference = reference.sort_values(['period', 'model']).reset_index(drop=True)
now = model_metrics.loc[model_metrics['model'].isin(['frozen', 'policy_b'])].copy()
now['model'] = now['model'].replace({'policy_b': 'updated'})
now = now.sort_values(['period', 'model']).reset_index(drop=True)
require(reference[['period', 'model']].equals(now[['period', 'model']]),
        '13B comparison cohorts or names differ.')
for col in ['loans', 'events', 'prevalence', 'roc_auc', 'average_precision',
            'brier', 'risk_gap', 'absolute_risk_gap', 'mean_predicted_risk']:
    np.testing.assert_allclose(now[col], reference[col], rtol=1e-8, atol=1e-10,
                               err_msg=f'Notebook 13B metric does not reproduce: {col}')
display(model_metrics.round(6))
display(point_comparisons.round(6))
print('Policy B and frozen metrics reproduce Notebook 13B. Positive improvement favors the first model.')


## 6. Use paired bootstrap intervals for all model comparisons

For each of 1,000 draws, resample the **same loans** for all three models. This isolates evaluation-sample uncertainty in their differences. It does not capture changes due to training random seeds, macroeconomic shocks, or historical data-release timing.

In [ ]:
BOOTSTRAP_DRAWS = 1000
rng = np.random.default_rng(42)
interval_rows = []
for period, part in GROUPS.items():
    y = part['y_true'].to_numpy(dtype=int)
    predictions = {name: part[f'p_{name}'].to_numpy(dtype=float)
                   for name in MODEL_NAMES}
    samples = {(first, second, metric): []
               for first, second in PAIRS for metric in METRICS}
    invalid = 0
    for _ in range(BOOTSTRAP_DRAWS):
        ix = rng.integers(0, len(y), size=len(y))
        sample_y = y[ix]
        if np.unique(sample_y).size != 2:
            invalid += 1
            continue
        values = {name: score_metrics(sample_y, p[ix])
                  for name, p in predictions.items()}
        for first, second in PAIRS:
            for metric in METRICS:
                a = values[first][metric]
                b = values[second][metric]
                samples[(first, second, metric)].append(
                    a-b if metric in HIGHER_BETTER else b-a
                )
    for (first, second, metric), values in samples.items():
        require(len(values) > 0, f'{period}: no valid bootstrap values.')
        low, high = np.quantile(values, [.025, .975])
        interval_rows.append({
            'period': period, 'comparison': f'{first}_minus_{second}',
            'metric': metric, 'ci_low': float(low), 'ci_high': float(high),
            'valid_draws': len(values), 'invalid_one_class_draws': invalid,
            'direction': ('favors_first' if low > 0 else
                          'favors_second' if high < 0 else 'uncertain'),
        })
    print('Bootstrap complete:', period, '| invalid one-class draws:', invalid)

intervals = pd.DataFrame(interval_rows)
paired = point_comparisons.merge(intervals,
    on=['period', 'comparison', 'metric'], how='inner', validate='one_to_one')
require(len(paired) == len(PAIRS)*len(METRICS)*len(GROUPS),
        'Missing one or more paired comparisons.')
require(paired['valid_draws'].ge(int(0.90 * BOOTSTRAP_DRAWS)).all(),
        'Too many bootstrap draws lacked both classes; review precision.')
display(paired.round(6))
print('Interpretation: the entire 95% interval must be above or below 0 to support a direction.')


## 7. Compare calibration on the same groups of loans

The ten groups below use 2023 **frozen-model scores** to assign loans. For a given group, we compare its actual event rate with all three models' mean predictions. The bins are descriptive only; they do not refit or recalibrate any model.

In [ ]:
common = evaluation.copy()
common['frozen_score_decile'] = pd.qcut(common['p_frozen'], 10,
                                           labels=False, duplicates='drop')
require(common['frozen_score_decile'].notna().all(), 'Unable to form frozen-score deciles.')
common['frozen_score_decile'] = common['frozen_score_decile'].astype(int)+1
common_calibration = (common.groupby('frozen_score_decile', sort=True)
    .agg(loans=('y_true', 'size'), events=('y_true', 'sum'),
         observed_rate=('y_true', 'mean'),
         frozen_mean=('p_frozen', 'mean'),
         policy_a_mean=('p_policy_a', 'mean'),
         policy_b_mean=('p_policy_b', 'mean')).reset_index())
require(len(common_calibration) == 10
        and int(common_calibration['loans'].sum()) == 49501
        and int(common_calibration['events'].sum()) == 876,
        'Frozen risk groups do not reconcile.')

fig_common, ax = plt.subplots(figsize=(8, 4.8))
for col, label in [('observed_rate','Observed event rate'),
                   ('frozen_mean','Frozen XGBoost'),
                   ('policy_a_mean','Policy A XGBoost'),
                   ('policy_b_mean','Policy B XGBoost')]:
    ax.plot(common_calibration['frozen_score_decile'],
            common_calibration[col]*100, marker='o', label=label)
ax.set_xlabel('Frozen-model score decile (low to high risk)')
ax.set_ylabel('Event rate / mean predicted risk (%)')
ax.set_xticks(common_calibration['frozen_score_decile'])
ax.set_title('2023: Three models compared on the same loan groups')
ax.grid(alpha=.2)
ax.legend()
fig_common.tight_layout()
display(common_calibration.round(6))
plt.show()


## 8. Save the Policy A model and an aggregate review bundle

The Policy A model is saved **locally in a new timestamped folder**, separate from both previous models. The review ZIP contains only aggregate metrics, cohort summaries, a calibration chart and provenance information. Do not upload `.joblib` files, raw Freddie Mac data or loan-level predictions to GitHub.

In [ ]:
# Recheck all original artifacts before writing new outputs.
require(sha256(MODEL_DATA) == base_info['input_file_sha256'], 'Modeling file changed during run.')
require(sha256(frozen_path) == base_info['models']['xgboost']['sha256'],
        'Frozen model changed during run.')
require(sha256(b_path) == b_info['updated_model_sha256'],
        'Policy B model changed during run.')
require(sha256(CONTRACT_PATH) == base_info['feature_contract_sha256'],
        'Feature contract changed during run.')

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = DATA / 'notebook_13_policy_a_sensitivity' / stamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
model_a_path = OUTPUT_DIR / 'freddie_xgboost_policy_A_jan2023.joblib'
joblib.dump(policy_a_model, model_a_path)
loaded_a = joblib.load(model_a_path)
np.testing.assert_allclose(
    loaded_a.predict_proba(X_2023.head(50))[:, 1],
    policy_a_model.predict_proba(X_2023.head(50))[:, 1],
    rtol=1e-12, atol=1e-12,
)

summary_files = {
    'policy_a_training_cohorts.csv': cohort_a,
    'evaluation_cohorts.csv': cohort_eval,
    'three_model_reliability_metrics.csv': model_metrics,
    'three_model_point_comparisons.csv': point_comparisons,
    'three_model_paired_intervals.csv': paired,
    'common_frozen_score_deciles.csv': common_calibration,
}
for filename, frame in summary_files.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)
fig_common.savefig(OUTPUT_DIR / 'three_model_common_calibration.png',
                   dpi=160, bbox_inches='tight')

manifest_a = {
    'stage': 'Notebook 13D Policy A sensitivity',
    'analysis_type': 'retrospective_exploratory_sensitivity',
    'decision_month': '2023-01',
    'policy_a_training_rule': '2015-2017 development plus fully mature 2018-H1 and 2018-H2',
    'training_loans': int(len(train_a)),
    'training_events': int(train_a[TARGET].sum()),
    'training_periods': sorted(train_a['period'].unique().tolist()),
    'policy_a_loan_keys_sha256': a_key_hash,
    'policy_audit_file': str(selected_audit.relative_to(ROOT)),
    'policy_b_training_loans': int(b_info['sample_rows']),
    'policy_b_training_events': int(b_info['sample_events']),
    'policy_b_stage2_run_id': RUN_ID_B,
    'frozen_model_sha256': sha256(frozen_path),
    'policy_b_model_sha256': sha256(b_path),
    'policy_a_model_sha256': sha256(model_a_path),
    'policy_a_model_filename': model_a_path.name,
    'feature_contract_sha256': sha256(CONTRACT_PATH),
    'model_data_sha256': sha256(MODEL_DATA),
    'predictor_count': len(PREDICTOR_COLUMNS),
    'encoded_feature_count': int(X_a_encoded.shape[1]),
    'xgboost_settings_unchanged': True,
    'original_fitted_preprocessing_reused': True,
    'evaluation_periods': ['2023-H1', '2023-H2'],
    'evaluation_loans': int(len(evaluation)),
    'evaluation_events': int(evaluation['y_true'].sum()),
    'paired_bootstrap_draws': BOOTSTRAP_DRAWS,
    'historical_publication_dates_verified': False,
    'independent_untouched_validation': False,
    'limitations': [
        'Historical reporting and target readiness relied on proxies, not verified release dates.',
        'Training membership was obtained from retrospectively eligible modeling loans.',
        '2023 evaluation outcomes had been examined before this sensitivity experiment.',
        'Loan-level bootstrap intervals omit training variation and correlated economic shocks.',
        'No validated operational retraining-trigger threshold or economic benefit is estimated.',
        'The 2023 frozen-score deciles are descriptive, not prospective monitoring cutoffs.',
    ],
}
(OUTPUT_DIR / 'policy_a_sensitivity_manifest.json').write_text(
    json.dumps(manifest_a, indent=2), encoding='utf-8'
)

zip_path = OUTPUT_DIR / 'notebook_13d_policy_a_review.zip'
review_names = [*summary_files, 'policy_a_sensitivity_manifest.json',
                'three_model_common_calibration.png']
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        archive.write(OUTPUT_DIR / name, arcname=name)
with zipfile.ZipFile(zip_path) as archive:
    require(archive.testzip() is None and len(archive.namelist()) == len(review_names),
            'Aggregate review ZIP integrity check failed.')

require(sha256(frozen_path) == base_info['models']['xgboost']['sha256']
        and sha256(b_path) == b_info['updated_model_sha256'],
        'An existing model changed during the run.')
print('Policy A model saved locally:', model_a_path)
print('Share aggregate review ZIP:', zip_path)
print('Frozen and Policy B artifacts remain unchanged. No loan-level data exported.')


## Review gate

Check the full 2023 and half-year results for all three models, then interpret the paired confidence intervals. Policy A may improve some metrics and worsen others; do **not** select a training policy solely because it looks best on the already examined 2023 sample.

**Workflow:** Run the notebook locally using the existing `.venv`. Save its executed results. Share `notebook_13d_policy_a_review.zip` for review first. We will push Notebook 13D to the GitHub feature branch only after reviewing the outputs. The next step after this sensitivity check is to document the complete retraining findings and limitations.